# Inference Callbacks

This notebook demonstrates the **callback system** introduced in Phase 4 of the inference pipeline refactor.
Callbacks hook into the inference lifecycle to provide timing, logging, safety checks, and other
instrumentation without modifying model or runner code.

We cover:
1. Baseline inference (no callbacks)
2. `TimingCallback` — measure prediction latency
3. `LoggingCallback` — structured lifecycle logging
4. Custom callback — clamp actions to safe limits
5. Custom callback — record input statistics
6. Composing multiple callbacks
7. Full lifecycle hooks (`on_load`, `on_predict_start`, `on_predict_end`, `on_reset`)
8. Context manager (`with` statement)

**Requirements:** No GPU or exported model needed — uses a mock adapter.

## Setup

Create a mock export directory and adapter so we can demonstrate callbacks without a real model.

In [1]:
from __future__ import annotations

import json
import tempfile
from pathlib import Path
from typing import Any, override
from unittest.mock import MagicMock, patch

import numpy as np
import yaml

from physicalai.inference import Callback, InferenceModel
from physicalai.inference.callbacks import LoggingCallback, TimingCallback


def _make_export_dir() -> Path:
    export_dir = Path(tempfile.mkdtemp(prefix="callback_demo_"))
    metadata = {
        "policy_class": "physicalai.policies.act.policy.ACT",
        "backend": "openvino",
        "use_action_queue": False,
        "chunk_size": 1,
    }
    with (export_dir / "metadata.yaml").open("w") as f:
        yaml.dump(metadata, f)
    manifest = {
        "format": "policy_package",
        "version": "1.0",
        "policy": {
            "name": "act",
            "kind": "single_pass",
            "class_path": "physicalai.policies.act.policy.ACT",
        },
        "artifacts": {"openvino": "act.xml"},
        "runner": {
            "class_path": "physicalai.inference.runners.SinglePass",
            "init_args": {},
        },
    }
    (export_dir / "manifest.json").write_text(json.dumps(manifest))
    (export_dir / "act.xml").touch()
    (export_dir / "act.bin").touch()
    return export_dir


def _make_adapter() -> MagicMock:
    adapter = MagicMock()
    adapter.input_names = ["state", "images"]
    adapter.output_names = ["actions"]
    adapter.predict.return_value = {
        "actions": np.random.default_rng(42).standard_normal((1, 10, 2)).astype(np.float32) * 100 + 150,
    }
    adapter.default_device.return_value = "cpu"
    return adapter


def _load_model(**kwargs: Any) -> InferenceModel:
    export_dir = _make_export_dir()
    adapter = _make_adapter()
    with patch("physicalai.inference.model.get_adapter", return_value=adapter):
        return InferenceModel(export_dir, **kwargs)


observation = {
    "state": np.array([[0.5, 0.3]], dtype=np.float32),
    "images": np.random.default_rng(0).standard_normal((1, 3, 96, 96)).astype(np.float32),
}

## 1. Baseline — No Callbacks

In [2]:
model = _load_model()
model.reset()
action = model.select_action(observation)

print(f"Model:       {model!r}")
print(f"Callbacks:   {model.callbacks}")
print(f"Action shape: {action.shape}")
print(f"Action dtype: {action.dtype}")

assert model.callbacks == []
assert action.shape[-1] == 2

Model:       InferenceModel(policy=act, backend=openvino, device=cpu, runner=SinglePass())
Callbacks:   []
Action shape: (1, 10, 2)
Action dtype: float32


## 2. TimingCallback — Measure Prediction Latency

In [3]:
timer = TimingCallback()
model = _load_model(callbacks=[timer])

for i in range(5):
    model.select_action(observation)

print(f"Last prediction: {timer.last_duration_ms:.3f} ms")
print(f"All durations:   {[f'{d:.3f}' for d in timer.history]}")
print(f"Average latency: {sum(timer.history) / len(timer.history):.3f} ms")
print(f"Min latency:     {min(timer.history):.3f} ms")
print(f"Max latency:     {max(timer.history):.3f} ms")
print(f"Timer repr:      {timer!r}")

assert len(timer.history) == 5
assert all(d > 0 for d in timer.history)

Last prediction: 0.009 ms
All durations:   ['0.025', '0.017', '0.012', '0.010', '0.009']
Average latency: 0.014 ms
Min latency:     0.009 ms
Max latency:     0.025 ms
Timer repr:      TimingCallback(last=0.0ms, calls=5)


## 3. LoggingCallback — Structured Lifecycle Logging

In [4]:
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(name)s | %(levelname)s | %(message)s",
    force=True,
)

logger_cb = LoggingCallback(level=logging.INFO)
model = _load_model(callbacks=[logger_cb])

model.select_action(observation)
model.reset()

print(f"Repr: {logger_cb!r}")

physicalai.inference | INFO | model_loaded | InferenceModel(policy=act, backend=openvino, device=cpu, runner=SinglePass())
physicalai.inference | INFO | predict_start | keys=['state', 'images']
physicalai.inference | INFO | predict_end | keys=['action']
physicalai.inference | INFO | reset


Repr: LoggingCallback(level=INFO)


## 4. Custom Callback — Clamp Actions to Safe Limits

Subclass `Callback` and override `on_predict_end` to modify outputs.

In [5]:
class SafetyClampCallback(Callback):
    """Clamp all actions to safe joint limits."""

    def __init__(self, low: float = -1.0, high: float = 1.0) -> None:
        self.low = low
        self.high = high
        self.clamp_count = 0

    @override
    def on_predict_end(self, outputs: dict[str, Any]) -> dict[str, Any]:
        action = outputs["action"]
        clamped = np.clip(action, self.low, self.high)
        if not np.array_equal(action, clamped):
            self.clamp_count += 1
        outputs["action"] = clamped
        return outputs

    def __repr__(self) -> str:
        return f"SafetyClampCallback(low={self.low}, high={self.high}, clamped={self.clamp_count})"


raw_model = _load_model()
raw_action = raw_model.select_action(observation)

safety = SafetyClampCallback(low=100.0, high=200.0)
clamped_model = _load_model(callbacks=[safety])
clamped_action = clamped_model.select_action(observation)

print(f"Raw action range:     [{raw_action.min():.2f}, {raw_action.max():.2f}]")
print(f"Clamped action range: [{clamped_action.min():.2f}, {clamped_action.max():.2f}]")
print(f"Times clamped:        {safety.clamp_count}")
print(f"Safety repr:          {safety!r}")

assert clamped_action.min() >= 100.0
assert clamped_action.max() <= 200.0

Raw action range:     [-45.10, 262.72]
Clamped action range: [100.00, 200.00]
Times clamped:        1
Safety repr:          SafetyClampCallback(low=100.0, high=200.0, clamped=1)


## 5. Custom Callback — Record Input Statistics

Override `on_predict_start` to inspect inputs. Return `None` to pass through unchanged.

In [6]:
class InputStatsCallback(Callback):
    """Record observation metadata before each prediction."""

    def __init__(self) -> None:
        self.call_count = 0
        self.last_keys: list[str] = []
        self.last_shapes: dict[str, tuple[int, ...]] = {}

    @override
    def on_predict_start(self, inputs: dict[str, Any]) -> None:
        self.call_count += 1
        self.last_keys = list(inputs.keys())
        self.last_shapes = {
            k: v.shape for k, v in inputs.items() if hasattr(v, "shape")
        }


stats = InputStatsCallback()
model = _load_model(callbacks=[stats])
model.select_action(observation)
model.select_action(observation)

print(f"Predictions recorded: {stats.call_count}")
print(f"Last input keys:      {stats.last_keys}")
print(f"Last input shapes:")
for k, v in stats.last_shapes.items():
    print(f"  {k}: {v}")

assert stats.call_count == 2

Predictions recorded: 2
Last input keys:      ['state', 'images']
Last input shapes:
  state: (1, 2)
  images: (1, 3, 96, 96)


## 6. Composing Multiple Callbacks

Pass a list of callbacks — they fire in declared order at each lifecycle point.

In [7]:
timer2 = TimingCallback()
safety2 = SafetyClampCallback(low=100.0, high=200.0)
logger_cb2 = LoggingCallback(level=logging.INFO)

model = _load_model(callbacks=[logger_cb2, timer2, safety2])

print(f"Registered: {model.callbacks}")
action = model.select_action(observation)

print(f"Action range: [{action.min():.2f}, {action.max():.2f}]")
print(f"Latency:      {timer2.last_duration_ms:.3f} ms")

assert action.min() >= 100.0
assert action.max() <= 200.0
assert timer2.last_duration_ms > 0.0

physicalai.inference | INFO | model_loaded | InferenceModel(policy=act, backend=openvino, device=cpu, runner=SinglePass())
physicalai.inference | INFO | predict_start | keys=['state', 'images']
physicalai.inference | INFO | predict_end | keys=['action']


Registered: [LoggingCallback(level=INFO), TimingCallback(last=0.0ms, calls=0), SafetyClampCallback(low=100.0, high=200.0, clamped=0)]
Action range: [100.00, 200.00]
Latency:      0.045 ms


## 7. Full Lifecycle Hooks

All four hooks fire in order: `on_load` → `on_predict_start` → `on_predict_end` → `on_reset`.

In [8]:
class LifecycleCallback(Callback):
    """Record all lifecycle events in order."""

    def __init__(self) -> None:
        self.events: list[str] = []

    @override
    def on_load(self, model: InferenceModel) -> None:
        self.events.append(f"loaded:{model.policy_name}")

    @override
    def on_predict_start(self, inputs: dict[str, Any]) -> None:
        self.events.append("predict_start")

    @override
    def on_predict_end(self, outputs: dict[str, Any]) -> None:
        self.events.append("predict_end")

    @override
    def on_reset(self) -> None:
        self.events.append("reset")


lifecycle = LifecycleCallback()
model = _load_model(callbacks=[lifecycle])
print(f"After load:    {lifecycle.events}")

model.select_action(observation)
print(f"After predict: {lifecycle.events}")

model.reset()
print(f"After reset:   {lifecycle.events}")

assert lifecycle.events == ["loaded:act", "predict_start", "predict_end", "reset"]

After load:    ['loaded:act']
After predict: ['loaded:act', 'predict_start', 'predict_end']
After reset:   ['loaded:act', 'predict_start', 'predict_end', 'reset']


## 8. Context Manager

In [9]:
timer3 = TimingCallback()

with _load_model(callbacks=[timer3]) as model:
    model.reset()
    action1 = model.select_action(observation)
    action2 = model.select_action(observation)

print(f"Predictions inside `with`: {len(timer3.history)}")
print(f"Durations: {[f'{d:.3f} ms' for d in timer3.history]}")
print(f"Action shapes: {action1.shape}, {action2.shape}")

assert len(timer3.history) == 2

Predictions inside `with`: 2
Durations: ['0.020 ms', '0.011 ms']
Action shapes: (1, 10, 2), (1, 10, 2)
